In [ ]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("1. Data load ho raha hai...")
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')
movies = movies.merge(credits, on='title')

print("2. Faltu data clean ho raha hai...")
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)

# Helper functions
def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]

def convert3(text):
    L = []
    for count, i in enumerate(ast.literal_eval(text)):
        if count < 3:
            L.append(i['name'])
        else:
            break
    return L

def fetch_director(text):
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            return [i['name']]
    return []

# Apply conversions
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Space removal logic (Sam Worthington fix)
for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(lambda x: [i.replace(" ", "") for i in x])

print("3. Tags create ho rahe hain...")
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())

print("4. Vectorization aur Math calculations chal rahi hain...")
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vectors)

1. Data load ho raha hai...
2. Faltu data clean ho raha hai...
3. Tags create ho rahe hain...
4. Vectorization aur Math calculations chal rahi hain...


In [42]:
def recommend(movie):
    if movie not in new_df['title'].values:
        print(f"Error: '{movie}' dataset me nahi mili. Spelling check karo.")
        return
        
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"\n--- Movies similar to '{movie}' ---")
    for i in movies_list:
        print("-", new_df.iloc[i[0]].title)

print("Pipeline Ready! ✅\n")

recommend('Interstellar')
recommend('Batman Begins')
recommend('The Lone Ranger')

Pipeline Ready! ✅


--- Movies similar to 'Interstellar' ---
- Silent Running
- Space Pirate Captain Harlock
- The Martian
- Apollo 13
- Moonraker

--- Movies similar to 'Batman Begins' ---
- The Dark Knight
- The Dark Knight Rises
- Batman
- Batman
- Batman & Robin

--- Movies similar to 'The Lone Ranger' ---
- The Wild Bunch
- Torque
- The Legend of the Lone Ranger
- West Side Story
- District B13
